# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

---
### TL;DR for reviewers

**Lane:** Lane 2 (Refresh/Content Opportunity Scoring) combined with Lane 4 (CTR/Engagement Opportunity Scoring), reframed as one diagnosis-first queue instead of two separate scores.

**The question:** when a page underperforms, *why* — genuine content decay, or the page getting answered on the SERP without a click — because those two need completely different fixes from an editor.

**Headline finding (Section 3, fully reproducible below):** **1,111 pages (3.7%)** show impressions holding or growing while clicks fall — and **0% of them** are caught by the current `trend_direction == 'down'` label. That's a real, currently-invisible blind spot in the existing approach, not an imported trend.

**Why it matters to FlyRank specifically:** industry data shows AI Overviews now cut organic CTR by roughly a third to two-thirds depending on query type, which means "traffic is down" has become ambiguous for every agency, not just FlyRank — a diagnosis-first queue turns an awkward client conversation into a defensible one.

**What I'm not claiming:** the pattern above is a *proxy signal*, not proof of AI-Overview cannibalization — this dataset has no query-level SERP-feature data to confirm cause. Section 4 spells this out in full.

---

## 1. My lane (or freestyle) and why

**Combined Lane 2 + Lane 4, as a diagnostic layer, not just a score.**

The starter pipeline (Lane 2) already ranks pages worth reviewing. Lane 4 separately asks which pages
underperform on clicks for their position. I want to merge these into one output that answers a sharper
question than either lane alone: **when a page underperforms, *why* - and does an editor actually need to
touch it?**

My reason for combining them is grounded in what's happening in search right now, not a stylistic
preference. Through 2025-2026, AI Overviews and zero-click search have made "traffic is down" ambiguous:
industry reporting describes agencies now needing to explain, every month, why impressions look fine but
clicks fell - because Google answered the query on the results page itself, not because the content got
worse. A flat "declining" queue conflates two problems that need completely different actions: genuine
content decay (an editor's job - refresh/rewrite) versus likely SERP-answer cannibalization (not an
editor's job at all - a schema/citability problem instead). Section 3 below shows this isn't just an
industry trend I'm importing - a small but real, currently-invisible segment of it already exists in
FlyRank's own starter data.

I'm treating this as a Lane 2/Lane 4 combination rather than a pure freestyle pick, since the guide allows
combining predefined lanes - but I'll confirm with my mentor early in Week 1 that the scope is reasonable
for 7 weeks, and I can fall back to a single lane if the diagnostic layer turns out to be too ambitious.

In [1]:
# Supporting check for Section 1: confirm the Lane 2 starter pipeline is a real, working reference
# (not just described in docs) before I claim I'm building on it.
import pandas as pd

queue = pd.read_csv('../../outputs/refresh_queue_sample.csv')
print(f"Existing Lane 2 queue: {len(queue):,} ranked pages already produced by the starter pipeline")
print(f"Suggested actions already assigned: {queue['suggested_action'].nunique()} distinct types")
print(queue['suggested_action'].value_counts())

Existing Lane 2 queue: 200 ranked pages already produced by the starter pipeline
Suggested actions already assigned: 3 distinct types
suggested_action
refresh_and_review_ctr           130
refresh                           35
refresh_and_review_engagement     35
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

**Research question:** *When a page's clicks or engagement fall short of what its search position should
produce, is that because the content itself needs work, or because the page is being answered directly on
the results page without a click - and does that distinction change what an editor should do?*

**Unit of analysis:** one content page (`content_id`), scored using its trailing-90-day and last-30d vs
prior-30d metrics.

**Decision it improves:** not just *which* pages to review, but *what kind of fix* a page needs before an
editor ever opens it. Today a flat "declining" list can't distinguish a page that needs a rewrite from one
that's fine but no longer gets clicked because the SERP answered the question already - so editors risk
spending time rewriting content that was never the problem.

**Who acts, and what do they do:** the same SEO editor/content strategist from Lane 2, at roughly 50
pages/week capacity - but now handed a *diagnosis*, not just a rank. Given a page tagged "genuine decline,"
they refresh or rewrite it. Given a page tagged "likely SERP-answered," they skip the rewrite and instead
flag it for a technical/schema review, or simply stop spending review time on it.

**Cost of a wrong call:** as before, I'm treating wasted editor time as the costlier error - specifically,
rewriting a page that was never going to recover clicks because the real cause is a SERP feature, not the
content. That's a new, sharper way to lose editor time that a plain "declining" list can't protect against.
The other error - mis-tagging a genuinely declining page as "SERP-answered" and leaving it alone - still
matters (a real decliner then goes unreviewed), so I'll report both error types honestly rather than
optimizing one away silently.

**Output:** a ranked queue like Lane 2's, but with an added diagnosis column (genuine decline / likely
SERP-answered / CTR-fixable / needs deeper look) and a suggested action per diagnosis, not just per page.

**Why data/ML helps, not just a rule:** the diagnosis depends on *how several signals move together*
(impressions trend, click trend, position, CTR relative to position) - exactly the kind of tangled,
multi-signal pattern where a learned model outperforms a single if-statement, as Lane 2's own baseline-vs-
random-forest comparison already demonstrates in Section 3. A follow-up check (see ML-03) confirms the same
shape for the CTR-fixable case specifically: no single column cleanly separates flagged pages from healthy
ones (the closest is content age, an 11-point gap) - the pattern is real but weak and spread across several
columns, which a fixed threshold can't combine but a model can.

In [2]:
# Supporting check for Section 2: put a real number on 'wasted editor time' (the false-positive cost),
# using the starter pipeline's own baseline-vs-model comparison already committed in outputs/model_report.md.
baseline_precision_at_50 = 0.240
rf_precision_at_50 = 0.740

print(f"Baseline rule: {1 - baseline_precision_at_50:.0%} of the top-50 flagged pages would be wasted editor time")
print(f"Random forest: {1 - rf_precision_at_50:.0%} of the top-50 flagged pages would be wasted editor time")
print("-> this is the concrete cost my 'false positives are the costlier error' framing is protecting against.")

Baseline rule: 76% of the top-50 flagged pages would be wasted editor time
Random forest: 26% of the top-50 flagged pages would be wasted editor time
-> this is the concrete cost my 'false positives are the costlier error' framing is protecting against.


## 3. Quick look at the data (2-3 real numbers)

*Loading `data/raw/content_refresh_anonymized.csv` directly below. First the Lane 2 scale/base-rate
numbers from before, then the new diagnostic-layer evidence: a candidate "likely SERP-answered" segment
that the current decline label misses entirely.*

In [3]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

n_pages = len(df)
n_clients = df['client_id'].nunique()
capacity_per_week = 50
weeks_to_cover_all = n_pages / capacity_per_week

declining_rate = (df['trend_direction'] == 'down').mean()
declining_count = (df['trend_direction'] == 'down').sum()

print(f"Pages in starter dataset: {n_pages:,} across {n_clients} clients")
print(f"At {capacity_per_week} pages/week review capacity, one full pass would take {weeks_to_cover_all:.0f} weeks")
print(f"Rows with trend_direction == 'down': {declining_count:,} ({declining_rate:.1%} of all rows)")
print("-> more than half the inventory looks 'declining' under the crude current-window label alone -")
print("   far too broad to hand an editor as-is. This is the gap a ranked, evidence-based score should close.")

Pages in starter dataset: 30,000 across 32 clients
At 50 pages/week review capacity, one full pass would take 600 weeks
Rows with trend_direction == 'down': 16,262 (54.2% of all rows)
-> more than half the inventory looks 'declining' under the crude current-window label alone -
   far too broad to hand an editor as-is. This is the gap a ranked, evidence-based score should close.


**The diagnostic segment, checked directly against the data (not assumed):**

In [4]:
# Candidate 'likely SERP-answered / AI-cannibalized' signature:
# impressions holding or growing, but clicks falling - a pattern distinct from genuine decline
# (where impressions AND clicks both fall together).
imp_change = df['impressions_last_30d'] - df['impressions_prev_30d']
clk_change = df['clicks_last_30d'] - df['clicks_prev_30d']

likely_serp_answered = df[(imp_change >= 0) & (clk_change < 0)]
genuine_decline = df[(imp_change < 0) & (clk_change < 0)]

print(f"Pages with impressions flat/up but clicks down: {len(likely_serp_answered):,} ({len(likely_serp_answered)/len(df):.1%})")
print(f"Pages with impressions AND clicks both down (genuine decline pattern): {len(genuine_decline):,} ({len(genuine_decline)/len(df):.1%})")
print(f"Share of the 'likely SERP-answered' group already caught by trend_direction=='down': {(likely_serp_answered.trend_direction=='down').mean():.0%}")
print()
print("-> a small but real segment (3.7% of pages) shows the 'impressions fine, clicks falling' pattern,")
print("   and NONE of it is caught by the current decline label - a genuine blind spot in the existing")
print("   approach, not a hypothetical one imported from industry reading.")

Pages with impressions flat/up but clicks down: 1,111 (3.7%)
Pages with impressions AND clicks both down (genuine decline pattern): 5,695 (19.0%)
Share of the 'likely SERP-answered' group already caught by trend_direction=='down': 0%

-> a small but real segment (3.7% of pages) shows the 'impressions fine, clicks falling' pattern,
   and NONE of it is caught by the current decline label - a genuine blind spot in the existing
   approach, not a hypothetical one imported from industry reading.


## 4. Careful words: what I can and can't claim

**What I can claim by the end of this project:**
- *Observed* associations between page-level signals (freshness, position, word count, engagement,
  impressions/click movement) and a defined outcome measured in the data.
- A *decision-support* ranking with a diagnosis attached: "this pattern looks most consistent with X,
  worth an editor's attention for reason Y" - not "this page is definitely being cannibalized by an AI
  Overview."
- A *comparison* against a transparent baseline rule, honestly validated with a held-out split (client
  holdout, since pages from the same client can share patterns a model could just memorize).

**What I will never claim:**
- That the "impressions flat/up, clicks down" pattern *proves* AI Overview or SERP-feature cannibalization.
  This dataset has no query-level SERP-feature data (whether an AI Overview appeared, whether the page was
  cited) - the pattern is a **proxy signature**, not a confirmed cause, and I'll say so every time I show it.
- That a refresh *caused* a recovery - that needs an experiment or a causal design, which this data alone
  cannot give me.
- That I discovered a Google ranking factor, or measured AI search rankings/citations - `ai_traffic_pct`
  and `ai_sessions_90d` are present in the schema but nonzero for only ~6% of rows in this starter slice,
  so I cannot model AI-referral share directly here; I can only use the impressions/clicks pattern as an
  indirect stand-in, and I'll name that limitation explicitly rather than let the diagnosis sound more
  certain than it is.
- That the starter label (`is_declining_label`, a current-window proxy) equals "this page will keep
  declining next month" - if I keep that label I'll say so as a known limitation; if I move to a
  future-window label (prior 90 days -> next 30 days), I'll run the full leakage checklist before trusting
  any result.
- That a high score or a diagnosis tag is "the truth" rather than one ranked opinion a human should still
  sanity-check - reason codes exist so a reviewer can see *why*, not just *that*.

In [5]:
# Supporting evidence for 'why ML, not just a rule' - the starter pipeline's own committed results.
import re

report = open('../../outputs/model_report.md').read()
print(report[report.find('## Model Comparison'):report.find('## Final Queue')].strip())
print()
print("Baseline rule Precision@50 = 0.240 (~12 of the top 50 flagged pages are true positives)")
print("Random forest Precision@50 = 0.740 (~37 of the top 50 flagged pages are true positives)")
print("-> a learned ranking meaningfully beats the fixed rule on this starter slice, which is the concrete")
print("   reason ML earns its place here rather than a hand-written if-statement.")

## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

Baseline rule Precision@50 = 0.240 (~12 of the top 50 flagged pages are true positives)
Random forest Precision@50 = 0.740 (~37 of the top 50 flagged pages are true positives)
-> a learned ranking meaningfully beats the fixed rule on this starter slice, which is the concrete
   reason ML earns its place here rather than a hand-written if-statement.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.